# Multi-Service Credential Management

**Scenario:** An analytics pipeline needs credentials for Google Analytics,
a PostgreSQL database, and an S3 bucket. Each credential lives in a different
backend: 1Password for service accounts, environment variables for CI,
Apple Keychain for local dev.

siege_utilities provides a unified `CredentialManager` that searches all
backends in priority order and reports what's available.

## 1. Credential Backend Discovery

Before configuring anything, check which backends are available on this
machine. The manager detects 1Password CLI, Apple Keychain, environment
variables, and filesystem credential files.

In [1]:
from siege_utilities.config.credential_manager import credential_status

backends = credential_status()

print(f"{'Backend':<15} {'Available':<12} {'Status'}")
print("-" * 55)
for name, info in backends.items():
    avail = 'Yes' if info['available'] else 'No'
    print(f"{name:<15} {avail:<12} {info['status']}")

[siege_utilities] 2026-06-03 11:20:21,272 INFO: Available credential backends: ['files', 'env', 'prompt', '1password', 'keychain']


[siege_utilities] 2026-06-03 11:20:21,273 INFO: Credential manager initialized with backends: {'files': True, 'env': True, 'prompt': True, '1password': True, 'keychain': True}


[siege_utilities] 2026-06-03 11:20:21,274 INFO: Credential search paths: [PosixPath('/Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/notebooks/config/credentials'), PosixPath('/Users/dheerajchand/.siege_utilities/credentials')]


Backend         Available    Status
-------------------------------------------------------
env             Yes          Always available
1password       Yes          Authenticated (5 accounts)
keychain        Yes          Available
prompt          Yes          Available (fallback)


## 2. The CredentialManager Class

The manager initializes with all available backends and provides a
unified `get_credential()` method. It searches backends in priority order:
1Password -> Keychain -> environment -> files -> interactive prompt.

When a credential isn't found, it raises `CredentialNotFoundError` —
never returns None or empty string (SU-1: errors are not data).

In [ ]:
from siege_utilities.config.credential_manager import (
    CredentialManager, CredentialNotFoundError
)

manager = CredentialManager()

print(f"Manager initialized")
print(f"Backends: {list(manager.backends.keys()) if hasattr(manager, 'backends') else 'configured'}")

# Missing credentials raise, not return empty
try:
    manager.get_credential("nonexistent-service", "fake-user")
except (CredentialNotFoundError, Exception) as e:
    print(f"\nMissing credential raises: {type(e).__name__}")
    print(f"  (SU-1: errors are not data)")

## 3. Environment Variable Pattern

In CI/CD, credentials come from environment variables. The manager
checks for them automatically. This is the pattern used in GitHub Actions
and Databricks secret scopes.

In [ ]:
import os

# Demonstrate the env-var credential pattern
# In production, these would be set by CI or the shell profile
os.environ["SIEGE_TEST_DB_HOST"] = "localhost"
os.environ["SIEGE_TEST_DB_PORT"] = "5432"

# The pattern: check env first, fall back to credential manager
db_host = os.environ.get("SIEGE_TEST_DB_HOST", "not-set")
db_port = os.environ.get("SIEGE_TEST_DB_PORT", "not-set")

print(f"DB Host: {db_host}")
print(f"DB Port: {db_port}")
print(f"\nPattern: os.environ.get() for CI, CredentialManager.get_credential() for local dev")

# Clean up
del os.environ["SIEGE_TEST_DB_HOST"]
del os.environ["SIEGE_TEST_DB_PORT"]

## Key Patterns

- **credential_status()** — discover available backends before configuring
- **CredentialManager** — unified search across 1Password, Keychain, env vars, files
- **CredentialNotFoundError** — missing credentials raise, never return empty (SU-1)
- **Priority order** — 1Password -> Keychain -> env -> files -> prompt
- **siege_zsh integration** — when available, shell profile pre-loads credentials
  into the environment so the manager finds them without interactive prompts